In [70]:
from pathlib import Path
from PIL import Image

import polars as pl

import torch
from torch.utils.data import Dataset
from torchvision import transforms

In [2]:
xray_dir = Path('datasets/nih-chest-xray/')
data_path = xray_dir / 'BBox_List_2017.csv'
data_entry = xray_dir / 'Data_Entry_2017.csv'
test_list_path = xray_dir / 'test_list.txt'
train_list_path = xray_dir / 'train_val_list.txt'

df1 = pl.read_csv(data_entry)
df2 = pl.read_csv(data_path)
df_test = pl.read_lines(test_list_path)
df_train = pl.read_lines(train_list_path)

In [3]:
df_train.to_series().head()

line
str
"""00000001_000.png"""
"""00000001_001.png"""
"""00000001_002.png"""
"""00000002_000.png"""
"""00000004_000.png"""
"""00000005_000.png"""
"""00000005_001.png"""
"""00000005_002.png"""
"""00000005_003.png"""


In [4]:
dirs = sorted(xray_dir.glob('images_0*/images/*.png'))
len(dirs)

24999

1. cardiomegaly
2. emphysema
3. effusion
4. hernia
5. infiltration
6. atelectasis
7. no findings
8. pneumonia
9. mass
10. nodule 
11. pneumothorax
12. pleural_thickening 
13. edema
14. consolidation 
15. fibrosis      

In [5]:
# df1.select(pl.col('Patient ID')).select(['Patient ID'])

train_corpus = df1.join(
    df_train, 
    left_on='Image Index',
    right_on='line',
    how='inner'
    )

test_corpus = df1.join(
    df_test,
    left_on='Image Index',
    right_on='line',
    how='inner'
).sort(['Patient ID', 'Follow-up #'])

intersect_df = train_corpus.join(
    test_corpus,
    on=['Image Index', 'Patient ID'],
    how='semi'
)

train_corpus[10:20]

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000005_005.png""","""No Finding""",5,5,70,"""F""","""PA""",2514,2991,0.143,0.143,null
"""00000005_006.png""","""Infiltration""",6,5,70,"""F""","""PA""",2992,2991,0.143,0.143,null
"""00000005_007.png""","""Effusion|Infiltration""",7,5,70,"""F""","""PA""",2566,2681,0.143,0.143,null
"""00000006_000.png""","""No Finding""",0,6,81,"""M""","""PA""",2500,2048,0.168,0.168,null
"""00000007_000.png""","""No Finding""",0,7,82,"""M""","""PA""",2500,2048,0.168,0.168,null
"""00000008_000.png""","""Cardiomegaly""",0,8,69,"""F""","""PA""",2048,2500,0.171,0.171,null
"""00000008_001.png""","""No Finding""",1,8,70,"""F""","""PA""",2048,2500,0.171,0.171,null
"""00000008_002.png""","""Nodule""",2,8,73,"""F""","""PA""",2048,2500,0.168,0.168,null
"""00000009_000.png""","""Emphysema""",0,9,73,"""M""","""PA""",2992,2991,0.143,0.143,null


In [6]:
train_corpus[77:87]

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000039_003.png""","""No Finding""",3,39,75,"""M""","""PA""",2464,2715,0.143,0.143,null
"""00000039_004.png""","""Effusion""",4,39,75,"""M""","""PA""",2718,2991,0.143,0.143,null
"""00000040_000.png""","""No Finding""",0,40,67,"""M""","""PA""",2500,2048,0.168,0.168,null
"""00000040_001.png""","""Emphysema""",1,40,67,"""M""","""AP""",2500,2048,0.168,0.168,null
"""00000040_002.png""","""No Finding""",2,40,67,"""M""","""AP""",2500,2048,0.168,0.168,null
"""00000040_003.png""","""Consolidation|Mass""",3,40,68,"""M""","""PA""",2500,2048,0.168,0.168,null
"""00000042_000.png""","""No Finding""",0,42,71,"""M""","""AP""",3056,2544,0.139,0.139,null
"""00000042_001.png""","""No Finding""",1,42,71,"""M""","""AP""",3056,2544,0.139,0.139,null
"""00000042_002.png""","""No Finding""",2,42,71,"""M""","""AP""",3056,2544,0.139,0.139,null


In [7]:
train_corpus.group_by("View Position").agg(
    pl.len(),
    pl.col("Finding Labels").mean().alias("anomaly_rate"),
)

View Position,len,anomaly_rate
str,u32,str
"""AP""",30310,null
"""PA""",56214,null


In [8]:
mask = train_corpus.select(pl.col('View Position'))
train_corpus.filter(pl.col('View Position') == 'AP')


Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000004_000.png""","""Mass|Nodule""",0,4,82,"""M""","""AP""",2500,2048,0.168,0.168,null
"""00000005_001.png""","""No Finding""",1,5,69,"""F""","""AP""",2500,2048,0.168,0.168,null
"""00000005_002.png""","""No Finding""",2,5,69,"""F""","""AP""",2500,2048,0.168,0.168,null
"""00000011_005.png""","""Infiltration""",5,11,75,"""M""","""AP""",2500,2048,0.168,0.168,null
"""00000015_000.png""","""No Finding""",0,15,70,"""M""","""AP""",2500,2048,0.168,0.168,null
…,…,…,…,…,…,…,…,…,…,…,…
"""00030786_003.png""","""Cardiomegaly|Effusion|Emphysem…",3,30786,61,"""F""","""AP""",3056,2544,0.139,0.139,null
"""00030786_004.png""","""Cardiomegaly|Effusion|Emphysem…",4,30786,61,"""F""","""AP""",3056,2544,0.139,0.139,null
"""00030786_005.png""","""Cardiomegaly|Effusion|Emphysem…",5,30786,61,"""F""","""AP""",3056,2544,0.139,0.139,null


In [9]:
test_corpus[24:34]

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000013_016.png""","""No Finding""",16,13,61,"""M""","""PA""",1725,2021,0.194311,0.194311,null
"""00000013_017.png""","""No Finding""",17,13,61,"""M""","""PA""",2021,2021,0.194311,0.194311,null
"""00000013_018.png""","""Effusion|Infiltration|Pneumoth…",18,13,60,"""M""","""AP""",3056,2544,0.139,0.139,null
"""00000013_019.png""","""No Finding""",19,13,62,"""M""","""PA""",2021,2021,0.194311,0.194311,null
"""00000013_020.png""","""Pneumothorax""",20,13,62,"""M""","""PA""",2021,2021,0.194311,0.194311,null
"""00000013_021.png""","""Effusion""",21,13,62,"""M""","""PA""",2021,2021,0.194311,0.194311,null
"""00000013_022.png""","""Infiltration|Mass""",22,13,56,"""M""","""PA""",2870,2991,0.143,0.143,null
"""00000013_023.png""","""Infiltration|Mass|Pneumothorax""",23,13,57,"""M""","""AP""",2500,2048,0.168,0.168,null
"""00000013_024.png""","""Mass""",24,13,57,"""M""","""AP""",2500,2048,0.168,0.168,null


In [10]:
torch.zeros(2, 1)


torch.tensor(1, dtype=torch.int16).shape

torch.Size([])

In [11]:
df1.head()

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000001_000.png""","""Cardiomegaly""",0,1,58,"""M""","""PA""",2682,2749,0.143,0.143,null
"""00000001_001.png""","""Cardiomegaly|Emphysema""",1,1,58,"""M""","""PA""",2894,2729,0.143,0.143,null
"""00000001_002.png""","""Cardiomegaly|Effusion""",2,1,58,"""M""","""PA""",2500,2048,0.168,0.168,null
"""00000002_000.png""","""No Finding""",0,2,81,"""M""","""PA""",2500,2048,0.171,0.171,null
"""00000003_000.png""","""Hernia""",0,3,81,"""F""","""PA""",2582,2991,0.143,0.143,null


In [12]:
print('df1 = ', len(df1))
df1.select(pl.col('Patient ID')).unique()

df1 =  112120


Patient ID
i64
18096
21767
24661
25185
2233
…
22282
1947
12436


In [13]:
df2.head()

Image Index,Finding Label,Bbox [x,y,w,h],,_duplicated_0,_duplicated_1
str,str,f64,f64,f64,f64,str,str,str
"""00013118_008.png""","""Atelectasis""",225.084746,547.019217,86.779661,79.186441,null,null,null
"""00014716_007.png""","""Atelectasis""",686.101695,131.543498,185.491525,313.491525,null,null,null
"""00029817_009.png""","""Atelectasis""",221.830508,317.053115,155.118644,216.949153,null,null,null
"""00014687_001.png""","""Atelectasis""",726.237288,494.95142,141.016949,55.322034,null,null,null
"""00017877_001.png""","""Atelectasis""",660.067797,569.780787,200.677966,78.101695,null,null,null


In [14]:
df1.sort(['Patient ID'])[844:854]

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000210_000.png""","""No Finding""",0,210,67,"""F""","""AP""",2500,2048,0.171,0.171,null
"""00000210_001.png""","""No Finding""",1,210,67,"""F""","""AP""",2500,2048,0.171,0.171,null
"""00000211_000.png""","""Cardiomegaly""",0,211,55,"""F""","""PA""",2048,2500,0.171,0.171,null
"""00000211_001.png""","""Cardiomegaly""",1,211,56,"""F""","""PA""",2500,2048,0.168,0.168,null
"""00000211_002.png""","""Cardiomegaly""",2,211,56,"""F""","""AP""",2500,2048,0.168,0.168,null
"""00000211_003.png""","""No Finding""",3,211,56,"""F""","""PA""",2048,2500,0.168,0.168,null
"""00000211_004.png""","""Cardiomegaly|Edema|Effusion|In…",4,211,58,"""F""","""PA""",2762,2589,0.143,0.143,null
"""00000211_005.png""","""Cardiomegaly|Infiltration""",5,211,58,"""F""","""PA""",2378,2497,0.143,0.143,null
"""00000211_006.png""","""Cardiomegaly|Edema|Effusion|Fi…",6,211,58,"""F""","""PA""",2048,2500,0.168,0.168,null


In [15]:
dd = df2.filter(pl.col('Finding Label') == 'Pneumonia')
dd

Image Index,Finding Label,Bbox [x,y,w,h],,_duplicated_0,_duplicated_1
str,str,f64,f64,f64,f64,str,str,str
"""00011136_002.png""","""Pneumonia""",574.008889,574.900156,229.831111,166.115556,null,null,null
"""00020259_002.png""","""Pneumonia""",614.968889,558.971267,223.004444,197.973333,null,null,null
"""00012021_081.png""","""Pneumonia""",660.48,190.331267,192.284444,451.697778,null,null,null
"""00013993_077.png""","""Pneumonia""",208.782222,403.095712,572.302222,392.533333,null,null,null
"""00013659_019.png""","""Pneumonia""",550.115556,463.397934,171.804444,275.342222,null,null,null
…,…,…,…,…,…,…,…,…
"""00012141_013.png""","""Pneumonia""",579.724868,209.134392,242.725926,379.259259,null,null,null
"""00022572_073.png""","""Pneumonia""",553.718519,328.330159,312.07619,416.101587,null,null,null
"""00003391_001.png""","""Pneumonia""",310.992593,242.725926,221.053968,360.838095,null,null,null


In [16]:
ls = {item for row in df1['Finding Labels'] for item in row.split('|')}
sorted(ls)

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'No Finding',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

In [17]:
classes = {
 'No Finding':0,
 'Atelectasis':1,
 'Cardiomegaly':2,
 'Consolidation':3,
 'Edema':4,
 'Effusion':5,
 'Emphysema':6,
 'Fibrosis':7,
 'Hernia':8,
 'Infiltration':9,
 'Mass':10,
 'Nodule':11,
 'Pleural_Thickening':12,
 'Pneumonia':13,
 'Pneumothorax':14
 }

classes

{'No Finding': 0,
 'Atelectasis': 1,
 'Cardiomegaly': 2,
 'Consolidation': 3,
 'Edema': 4,
 'Effusion': 5,
 'Emphysema': 6,
 'Fibrosis': 7,
 'Hernia': 8,
 'Infiltration': 9,
 'Mass': 10,
 'Nodule': 11,
 'Pleural_Thickening': 12,
 'Pneumonia': 13,
 'Pneumothorax': 14}

In [18]:
items = dd.select('Image Index')
# ss = set(items)
it = iter(items.to_series())
set(it)

{'00000150_002.png',
 '00000193_019.png',
 '00000506_013.png',
 '00001933_000.png',
 '00002711_000.png',
 '00002980_000.png',
 '00003028_006.png',
 '00003064_035.png',
 '00003391_001.png',
 '00003528_024.png',
 '00003789_000.png',
 '00003894_005.png',
 '00004344_002.png',
 '00004808_090.png',
 '00004893_070.png',
 '00005353_000.png',
 '00005532_022.png',
 '00005567_000.png',
 '00005567_025.png',
 '00005869_001.png',
 '00007444_003.png',
 '00007629_001.png',
 '00007728_013.png',
 '00007735_018.png',
 '00008291_009.png',
 '00008291_011.png',
 '00008727_009.png',
 '00009107_006.png',
 '00009229_003.png',
 '00009507_004.png',
 '00009608_037.png',
 '00009863_058.png',
 '00010120_010.png',
 '00010277_000.png',
 '00010447_018.png',
 '00010610_003.png',
 '00010652_000.png',
 '00010828_023.png',
 '00011136_002.png',
 '00011237_006.png',
 '00011366_002.png',
 '00011402_007.png',
 '00011514_015.png',
 '00011583_000.png',
 '00011831_008.png',
 '00012021_081.png',
 '00012045_005.png',
 '00012094_01

In [19]:
x = dd.columns[2]
h = dd.columns[5]

dd.select(['Bbox [x', 'y', 'w', 'h]'])

Bbox [x,y,w,h]
f64,f64,f64,f64
574.008889,574.900156,229.831111,166.115556
614.968889,558.971267,223.004444,197.973333
660.48,190.331267,192.284444,451.697778
208.782222,403.095712,572.302222,392.533333
550.115556,463.397934,171.804444,275.342222
…,…,…,…
579.724868,209.134392,242.725926,379.259259
553.718519,328.330159,312.07619,416.101587
310.992593,242.725926,221.053968,360.838095


In [20]:
list(dd.columns)

['Image Index',
 'Finding Label',
 'Bbox [x',
 'y',
 'w',
 'h]',
 '',
 '_duplicated_0',
 '_duplicated_1']

In [51]:
items = sorted(xray_dir.glob('images_0*/images/*.png'))
items[:5]

[PosixPath('datasets/nih-chest-xray/images_001/images/00000001_000.png'),
 PosixPath('datasets/nih-chest-xray/images_001/images/00000001_001.png'),
 PosixPath('datasets/nih-chest-xray/images_001/images/00000001_002.png'),
 PosixPath('datasets/nih-chest-xray/images_001/images/00000002_000.png'),
 PosixPath('datasets/nih-chest-xray/images_001/images/00000003_000.png')]

In [55]:
str(items[0])

'datasets/nih-chest-xray/images_001/images/00000001_000.png'

In [62]:
urls = [(p.name, str(p)) for p in items]

df_path = pl.DataFrame(
    urls,
    schema=['Image Index', 'Image Path'],
    orient='row'
)

train_data = df1.join(
    df_path,
    on='Image Index',
    how='inner'
)

train_data.head()

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],,Image Path
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str,str
"""00000001_000.png""","""Cardiomegaly""",0,1,58,"""M""","""PA""",2682,2749,0.143,0.143,null,"""datasets/nih-chest-xray/images…"
"""00000001_001.png""","""Cardiomegaly|Emphysema""",1,1,58,"""M""","""PA""",2894,2729,0.143,0.143,null,"""datasets/nih-chest-xray/images…"
"""00000001_002.png""","""Cardiomegaly|Effusion""",2,1,58,"""M""","""PA""",2500,2048,0.168,0.168,null,"""datasets/nih-chest-xray/images…"
"""00000002_000.png""","""No Finding""",0,2,81,"""M""","""PA""",2500,2048,0.171,0.171,null,"""datasets/nih-chest-xray/images…"
"""00000003_000.png""","""Hernia""",0,3,81,"""F""","""PA""",2582,2991,0.143,0.143,null,"""datasets/nih-chest-xray/images…"


In [63]:
train_data.slice(7, 1).get_column('Image Path').item()

'datasets/nih-chest-xray/images_001/images/00000003_003.png'

In [81]:
class XRayData(Dataset):

    def __init__(self, corpus, transform=None):
        super().__init__()
        self.corpus = corpus
        self.transform = transform
        self.transform = transform or transforms.Compose([
                    transforms.Resize((256, 256)),
                    transforms.ToTensor(),
                    transforms.Normalize([0.5]*3, [0.5]*3)
                ])

    def __len__(self):
        return len(self.corpus)

    def __getitem__(self, index):
        row = self.corpus.slice(index, 1)
        p = self.corpus.slice(index, 1).get_column('Image Path').item()
        image = Image.open(Path(p)).convert('RGB')
        return self.transform(image)



In [82]:
dset = XRayData(train_data)

In [84]:
dset[7]

tensor([[[-1.0000, -0.8039, -0.7098,  ..., -0.8902, -0.7333, -0.4196],
         [-1.0000, -0.8039, -0.7176,  ..., -0.9137, -0.7882, -0.5137],
         [-0.9922, -0.7804, -0.7255,  ..., -0.9137, -0.7961, -0.5294],
         ...,
         [ 0.1216, -0.0667, -0.2863,  ...,  0.9843,  0.1216, -0.9765],
         [ 0.2549,  0.0667, -0.1529,  ...,  0.9843,  0.0118, -0.9922],
         [ 0.4196,  0.2235,  0.0275,  ...,  0.9843, -0.0353, -1.0000]],

        [[-1.0000, -0.8039, -0.7098,  ..., -0.8902, -0.7333, -0.4196],
         [-1.0000, -0.8039, -0.7176,  ..., -0.9137, -0.7882, -0.5137],
         [-0.9922, -0.7804, -0.7255,  ..., -0.9137, -0.7961, -0.5294],
         ...,
         [ 0.1216, -0.0667, -0.2863,  ...,  0.9843,  0.1216, -0.9765],
         [ 0.2549,  0.0667, -0.1529,  ...,  0.9843,  0.0118, -0.9922],
         [ 0.4196,  0.2235,  0.0275,  ...,  0.9843, -0.0353, -1.0000]],

        [[-1.0000, -0.8039, -0.7098,  ..., -0.8902, -0.7333, -0.4196],
         [-1.0000, -0.8039, -0.7176,  ..., -0

In [23]:
r = df1.slice(25, 1)
f = r.get_column('Finding Labels').item()
type(f)

str

In [24]:
dd.select('Bbox [x')

Bbox [x
f64
574.008889
614.968889
660.48
208.782222
550.115556
…
579.724868
553.718519
310.992593


In [25]:
[p for p in dirs if p.name in items]

[]

In [26]:
df1.head()

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000001_000.png""","""Cardiomegaly""",0,1,58,"""M""","""PA""",2682,2749,0.143,0.143,null
"""00000001_001.png""","""Cardiomegaly|Emphysema""",1,1,58,"""M""","""PA""",2894,2729,0.143,0.143,null
"""00000001_002.png""","""Cardiomegaly|Effusion""",2,1,58,"""M""","""PA""",2500,2048,0.168,0.168,null
"""00000002_000.png""","""No Finding""",0,2,81,"""M""","""PA""",2500,2048,0.171,0.171,null
"""00000003_000.png""","""Hernia""",0,3,81,"""F""","""PA""",2582,2991,0.143,0.143,null


In [27]:
labels = df1.select(pl.col('Finding Labels')).unique()
labels

Finding Labels
str
"""Effusion|Pneumothorax|Mass"""
"""Infiltration|Pleural_Thickenin…"
"""Effusion|Mass|Nodule|Pneumotho…"
"""Atelectasis|Consolidation"""
"""Atelectasis|Cardiomegaly|Conso…"
…
"""Consolidation|Emphysema|Infilt…"
"""Consolidation|Fibrosis"""
"""Cardiomegaly|Edema|Infiltratio…"


In [28]:
labels[1]

Finding Labels
str
"""Infiltration|Pleural_Thickenin…"


In [29]:
print(len(df1.columns))
print(type(df1.columns))
print(df1.columns)
df1.columns

12
<class 'list'>
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', '']


['Image Index',
 'Finding Labels',
 'Follow-up #',
 'Patient ID',
 'Patient Age',
 'Patient Gender',
 'View Position',
 'OriginalImage[Width',
 'Height]',
 'OriginalImagePixelSpacing[x',
 'y]',
 '']

In [30]:
df2.columns

['Image Index',
 'Finding Label',
 'Bbox [x',
 'y',
 'w',
 'h]',
 '',
 '_duplicated_0',
 '_duplicated_1']

In [31]:
print('df1: ', len(df1))
print('df2: ', len(df2))

df1:  112120
df2:  984


In [32]:
df2.select('Finding Label').unique()

Finding Label
str
"""Infiltrate"""
"""Atelectasis"""
"""Pneumothorax"""
"""Cardiomegaly"""
"""Effusion"""
"""Mass"""
"""Nodule"""
"""Pneumonia"""


In [33]:
df2.filter(pl.col('Finding Label') == 'No Finding')

Image Index,Finding Label,Bbox [x,y,w,h],,_duplicated_0,_duplicated_1
str,str,f64,f64,f64,f64,str,str,str


In [34]:
df2.select(pl.col('Finding Label')).unique()

Finding Label
str
"""Mass"""
"""Nodule"""
"""Effusion"""
"""Cardiomegaly"""
"""Atelectasis"""
"""Infiltrate"""
"""Pneumonia"""
"""Pneumothorax"""


In [35]:
df2.row(1)

('00014716_007.png',
 'Atelectasis',
 686.101694915254,
 131.543498411017,
 185.491525423729,
 313.491525423729,
 None,
 None,
 None)